In [1]:
import jax
from jax import numpy as jnp

In [34]:
point_1 = jnp.zeros((4,2))
point_2 = jnp.array([[0,1],[0,2],[1,0],[1,1]])
diff = point_1 - point_2
jnp.sqrt(jnp.expand_dims(diff, -2) @ jnp.expand_dims(diff, -1))[:,0]

Array([[1.       ],
       [2.       ],
       [1.       ],
       [1.4142135]], dtype=float32)

In [42]:
point_1 = jnp.zeros((4,2))
point_2 = jnp.array([[0,1],[0,2],[1,0],[1,1]])
diff = point_1 - point_2
jnp.sqrt(jnp.sum(diff ** 2, axis=-1))

Array([1.       , 2.       , 1.       , 1.4142135], dtype=float32)

In [35]:
point_1 = jnp.zeros((2,))
point_2 = jnp.array([0,2])
diff = point_1 - point_2
jnp.sqrt(jnp.expand_dims(diff, -2) @ jnp.expand_dims(diff, -1))[:,0]

Array([2.], dtype=float32)

In [36]:
key = jax.random.PRNGKey(0)
key

Array([0, 0], dtype=uint32)

In [38]:
jax.random.split(key)

Array([[1797259609, 2579123966],
       [ 928981903, 3453687069]], dtype=uint32)

In [40]:
import random
import math

def distance(p1, p2):
    """Euclidean distance between two points."""
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)

BOARD_SIZE = 100.0            # Board is a continuous 100x100 2D space.
CENTER = BOARD_SIZE / 2.0     # The center of the board is (50, 50).
SUN_RADIUS = 10.0             # Fleets that pass within this distance to the center are destroyed.
ROTATION_RADIUS_LIMIT = 50.0  # Planets inside this limit revolve around the sun. Planets outside are static.
COMET_RADIUS = 1.0            # Radius of extra-solar temporary planets (comets).
COMET_PRODUCTION = 1          # Comets produce 1 ship per turn.
PLANET_CLEARANCE = 7          # Planets must be at least this far apart from each other.
MIN_PLANET_GROUPS = 5         # Minimum number of 4-planet groups (20 planets).
MAX_PLANET_GROUPS = 10        # Maximum number of 4-planet groups (40 planets).
MIN_STATIC_GROUPS = 3         # Minimum number of static (non-revolving) 4-planet groups (12 planets).
COMET_SPAWN_STEPS = [50, 150, 250, 350, 450] # Specific turns where comets spawn.

def generate_planets(rng=None):
    """
    Generates planets with 4-fold rotational symmetry.
    This guarantees the game is perfectly fair for 2 or 4 players,
    as every quadrant is identical to the others rotated by 90 degrees.
    """
    if rng is None:
        rng = random
    planets = []
    num_q1 = rng.randint(MIN_PLANET_GROUPS, MAX_PLANET_GROUPS)
    id_counter = 0

    # Phase 1: Generate guaranteed STATIC planet groups (orbital distance + radius >= 50).
    static_groups = 0
    for _ in range(5000):
        if static_groups >= MIN_STATIC_GROUPS:
            break
        prod = rng.randint(1, 5)
        r = 1 + math.log(prod)
        angle = rng.uniform(0, math.pi / 2)  # Limit initial spawn to Quadrant 1 (Q1)

        min_orbital = ROTATION_RADIUS_LIMIT - r
        max_orbital = (BOARD_SIZE - CENTER - r) / max(math.cos(angle), math.sin(angle))
        if min_orbital > max_orbital:
            continue

        orbital_r = rng.uniform(min_orbital, max_orbital)
        x = CENTER + orbital_r * math.cos(angle)
        y = CENTER + orbital_r * math.sin(angle)

        # Validate board boundaries for all 4 symmetric reflections
        if x + r > BOARD_SIZE or x - r < 0 or y + r > BOARD_SIZE or y - r < 0:
            continue
        if (BOARD_SIZE - x) - r < 0 or (BOARD_SIZE - y) - r < 0:
            continue

        # Ensure they don't spawn overlapping the axes to maintain perfect symmetry
        if (x - CENTER) < r + 5 or (y - CENTER) < r + 5:
            continue

        ships = min(rng.randint(5, 99), rng.randint(5, 99))

        # 4 symmetric planets: Q1, Q2, Q3, Q4
        temp_planets = [
            [id_counter, -1, y, x, r, ships, prod],
            [id_counter + 1, -1, BOARD_SIZE - x, y, r, ships, prod],
            [id_counter + 2, -1, x, BOARD_SIZE - y, r, ships, prod],
            [id_counter + 3, -1, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod],
        ]

        # Check overlap with existing planets
        valid = True
        for tp in temp_planets:
            for p in planets:
                if distance((p[2], p[3]), (tp[2], tp[3])) < p[4] + tp[4] + PLANET_CLEARANCE:
                    valid = False
                    break
            if not valid:
                break

        if valid:
            planets.extend(temp_planets)
            id_counter += 4
            static_groups += 1

    # Phase 2: Generate remaining planet groups (can be orbiting or static).
    attempts = 0
    max_attempts = 5000
    has_orbiting = False

    while len(planets) < num_q1 * 4 or (not has_orbiting and attempts < max_attempts):
        attempts += 1
        if attempts >= max_attempts:
            break
        prod = rng.randint(1, 5)
        r = 1 + math.log(prod)
        x = rng.uniform(CENTER + 15, BOARD_SIZE - r - 5)
        y = rng.uniform(CENTER + 15, BOARD_SIZE - r - 5)

        orbital_radius = distance((x, y), (CENTER, CENTER))

        # Reject if spawned inside or touching the Sun
        if orbital_radius < SUN_RADIUS + r + 10:
            continue

        # If static, make sure it's fully inside the board
        if orbital_radius + r >= ROTATION_RADIUS_LIMIT:
            if x + r > BOARD_SIZE or x - r < 0 or y + r > BOARD_SIZE or y - r < 0:
                continue

        valid = True
        ships = rng.randint(5, 30)
        temp_planets = [
            [id_counter, -1, y, x, r, ships, prod],
            [id_counter + 1, -1, BOARD_SIZE - x, y, r, ships, prod],
            [id_counter + 2, -1, x, BOARD_SIZE - y, r, ships, prod],
            [id_counter + 3, -1, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod],
        ]

        # Sweeping overlap check to ensure an orbiting planet won't collide with a static one
        # at any point in its orbit.
        for tp in temp_planets:
            tp_orbital = distance((tp[2], tp[3]), (CENTER, CENTER))
            tp_is_rotating = tp_orbital + tp[4] < ROTATION_RADIUS_LIMIT

            for p in planets:
                p_orbital = distance((p[2], p[3]), (CENTER, CENTER))
                p_is_rotating = p_orbital + p[4] < ROTATION_RADIUS_LIMIT

                if distance((p[2], p[3]), (tp[2], tp[3])) < p[4] + tp[4] + PLANET_CLEARANCE:
                    valid = False
                    break

                # If one is rotating and one is static, check their minimum distance
                # over the entire rotation path using absolute difference in orbital radii
                if tp_is_rotating != p_is_rotating:
                    if abs(tp_orbital - p_orbital) < tp[4] + p[4] + PLANET_CLEARANCE:
                        valid = False
                        break

            if not valid:
                break

        if valid:
            if orbital_radius + r < ROTATION_RADIUS_LIMIT:
                has_orbiting = True
            planets.extend(temp_planets)
            id_counter += 4

    return planets

In [41]:
generate_planets()

[[0, -1, 65.25945581790717, 95.19118588561852, 2.386294361119891, 10, 4],
 [1, -1, 4.808814114381477, 65.25945581790717, 2.386294361119891, 10, 4],
 [2, -1, 95.19118588561852, 34.74054418209283, 2.386294361119891, 10, 4],
 [3, -1, 34.74054418209283, 4.808814114381477, 2.386294361119891, 10, 4],
 [4, -1, 90.2370017992938, 81.00110562785156, 2.6094379124341005, 40, 5],
 [5, -1, 18.998894372148442, 90.2370017992938, 2.6094379124341005, 40, 5],
 [6, -1, 81.00110562785156, 9.762998200706207, 2.6094379124341005, 40, 5],
 [7, -1, 9.762998200706207, 18.998894372148442, 2.6094379124341005, 40, 5],
 [8, -1, 98.84951818481866, 71.23053594571887, 1.0, 19, 1],
 [9, -1, 28.76946405428113, 98.84951818481866, 1.0, 19, 1],
 [10, -1, 71.23053594571887, 1.1504818151813367, 1.0, 19, 1],
 [11, -1, 1.1504818151813367, 28.76946405428113, 1.0, 19, 1],
 [12, -1, 68.9196214023188, 78.42093000022078, 1.0, 7, 1],
 [13, -1, 21.57906999977922, 68.9196214023188, 1.0, 7, 1],
 [14, -1, 78.42093000022078, 31.0803785976

In [49]:
import jax.random as jrandom
from jax import lax
def distance(p1, p2):
    """Euclidean distance between two points. Can be batched."""
    diff = p1 - p2
    return jnp.sqrt(jnp.sum(diff ** 2, axis=-1))

def generate_planets(rng_key):
    """
    Generates planets with 4-fold rotational symmetry in a JAX-friendly way.
    Uses while loops and fixed-size arrays to ensure XLA compilation.
    Returns:
        planets: [MAX_PLANET_GROUPS * 4, 7] array of planets.
                 Fields: [id, owner, y, x, r, ships, production]
                 Empty slots have id = -1.
    """
    rng_key, subkey = jrandom.split(rng_key)

    # Decide number of Q1 planets
    num_q1 = jrandom.randint(subkey, (), MIN_PLANET_GROUPS, MAX_PLANET_GROUPS + 1)
    max_total_planets = MAX_PLANET_GROUPS * 4

    # Initialize state for Phase 1 (Static planets)
    # State holds: (planets_array, num_planets, static_groups, rng_key, attempts)
    init_state_1 = (
        jnp.full((max_total_planets, 7), -1.0),
        0,
        0,
        rng_key,
        0
    )

    def phase_1_cond(state):
        _, _, static_groups, _, attempts = state
        return (static_groups < MIN_STATIC_GROUPS) & (attempts < 5000)

    def phase_1_body(state):
        planets, num_planets, static_groups, key, attempts = state
        key, k1, k2, k3, k4 = jrandom.split(key, 5)

        prod = jrandom.randint(k1, (), 1, 6)
        r = 1.0 + jnp.log(prod)
        angle = jrandom.uniform(k2, (), minval=0.0, maxval=jnp.pi / 2.0)

        min_orbital = ROTATION_RADIUS_LIMIT - r
        max_orbital = (BOARD_SIZE - CENTER - r) / jnp.maximum(jnp.cos(angle), jnp.sin(angle))

        # Check valid bounds for generation
        valid_bounds = min_orbital <= max_orbital

        orbital_r = jrandom.uniform(k3, (), minval=min_orbital, maxval=jnp.maximum(min_orbital, max_orbital))
        x = CENTER + orbital_r * jnp.cos(angle)
        y = CENTER + orbital_r * jnp.sin(angle)

        # Check conditions
        cond1 = (x + r <= BOARD_SIZE) & (x - r >= 0) & (y + r <= BOARD_SIZE) & (y - r >= 0)
        cond2 = ((BOARD_SIZE - x) - r >= 0) & ((BOARD_SIZE - y) - r >= 0)
        cond3 = ((x - CENTER) >= r + 5.0) & ((y - CENTER) >= r + 5.0)

        valid_generation = valid_bounds & cond1 & cond2 & cond3

        ships = jnp.minimum(jrandom.randint(k4, (), 5, 100), jrandom.randint(k4, (), 5, 100)) # Simple mock for min(rand, rand)

        id_base = num_planets

        tp = jnp.array([
            [id_base,     -1.0, y, x, r, ships, prod],
            [id_base + 1, -1.0, BOARD_SIZE - x, y, r, ships, prod],
            [id_base + 2, -1.0, x, BOARD_SIZE - y, r, ships, prod],
            [id_base + 3, -1.0, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod]
        ])

        # Distance checks
        # Create pairwise distance mask. We only care about populated planets (i < num_planets)
        def check_overlap(new_planet, planets, num_planets):
            # Check all existing planets
            dists = distance(jnp.array([new_planet[2], new_planet[3]]), planets[:, 2:4])
            min_dist_req = new_planet[4] + planets[:, 4] + PLANET_CLEARANCE
            # Only active planets
            active_mask = jnp.arange(max_total_planets) < num_planets
            overlap = active_mask & (dists < min_dist_req)
            return jnp.any(overlap)

        overlap1 = check_overlap(tp[0], planets, num_planets)
        overlap2 = check_overlap(tp[1], planets, num_planets)
        overlap3 = check_overlap(tp[2], planets, num_planets)
        overlap4 = check_overlap(tp[3], planets, num_planets)

        has_overlap = overlap1 | overlap2 | overlap3 | overlap4

        success = valid_generation & (~has_overlap)

        new_planets = lax.cond(
            success,
            lambda p: lax.dynamic_update_slice(p, tp, (num_planets, 0)),
            lambda p: p,
            planets
        )

        new_num = num_planets + jnp.where(success, 4, 0)
        new_static = static_groups + jnp.where(success, 1, 0)

        return (new_planets, new_num, new_static, key, attempts + 1)

    state_after_1 = lax.while_loop(phase_1_cond, phase_1_body, init_state_1)

    # Phase 2
    planets, num_planets, _, rng_key, _ = state_after_1

    init_state_2 = (
        planets,
        num_planets,
        False, # has_orbiting
        rng_key,
        0      # attempts
    )

    def phase_2_cond(state):
        _, num_planets, has_orbiting, _, attempts = state
        target_planets = num_q1 * 4
        # Need to reach target planets, OR we need at least one orbiting planet
        needs_more = num_planets < target_planets
        needs_orbiting = (~has_orbiting)
        return (needs_more | needs_orbiting) & (attempts < 5000) & (num_planets < max_total_planets)

    def phase_2_body(state):
        planets, num_planets, has_orbiting, key, attempts = state
        key, k1, k2, k3 = jrandom.split(key, 4)

        prod = jrandom.randint(k1, (), 1, 6)
        r = 1.0 + jnp.log(prod)
        x = jrandom.uniform(k2, (), minval=CENTER + 15.0, maxval=BOARD_SIZE - r - 5.0)
        y = jrandom.uniform(k3, (), minval=CENTER + 15.0, maxval=BOARD_SIZE - r - 5.0)

        orbital_radius = distance(jnp.array([x, y]), jnp.array([CENTER, CENTER]))

        cond_sun = orbital_radius >= (SUN_RADIUS + r + 10.0)

        is_static = (orbital_radius + r) >= ROTATION_RADIUS_LIMIT
        cond_static_bounds = jnp.where(
            is_static,
            (x + r <= BOARD_SIZE) & (x - r >= 0) & (y + r <= BOARD_SIZE) & (y - r >= 0),
            True
        )

        valid_generation = cond_sun & cond_static_bounds

        ships = jrandom.randint(key, (), 5, 31)

        id_base = num_planets
        tp = jnp.array([
            [id_base,     -1.0, y, x, r, ships, prod],
            [id_base + 1, -1.0, BOARD_SIZE - x, y, r, ships, prod],
            [id_base + 2, -1.0, x, BOARD_SIZE - y, r, ships, prod],
            [id_base + 3, -1.0, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod]
        ])

        def check_phase2_overlap(new_planet, planets, num_planets):
            tp_orbital = distance(jnp.array([new_planet[2], new_planet[3]]), jnp.array([CENTER, CENTER]))
            tp_is_rotating = (tp_orbital + new_planet[4]) < ROTATION_RADIUS_LIMIT

            p_orbitals = distance(planets[:, 2:4], jnp.array([CENTER, CENTER]))
            p_is_rotating = (p_orbitals + planets[:, 4]) < ROTATION_RADIUS_LIMIT

            dists = distance(jnp.array([new_planet[2], new_planet[3]]), planets[:, 2:4])
            min_dist_req = new_planet[4] + planets[:, 4] + PLANET_CLEARANCE

            overlap_standard = dists < min_dist_req

            # Cross check for static vs rotating
            cross_diff = jnp.abs(tp_orbital - p_orbitals)
            overlap_cross = (tp_is_rotating != p_is_rotating) & (cross_diff < min_dist_req)

            overlap_total = overlap_standard | overlap_cross
            active_mask = jnp.arange(max_total_planets) < num_planets
            return jnp.any(active_mask & overlap_total)

        overlap1 = check_phase2_overlap(tp[0], planets, num_planets)
        overlap2 = check_phase2_overlap(tp[1], planets, num_planets)
        overlap3 = check_phase2_overlap(tp[2], planets, num_planets)
        overlap4 = check_phase2_overlap(tp[3], planets, num_planets)

        has_overlap = overlap1 | overlap2 | overlap3 | overlap4

        success = valid_generation & (~has_overlap)

        new_planets = lax.cond(
            success,
            lambda p: lax.dynamic_update_slice(p, tp, (num_planets, 0)),
            lambda p: p,
            planets
        )

        new_num = num_planets + jnp.where(success, 4, 0)

        # Check if the generated group has orbiting planets
        tp_orbital = distance(jnp.array([x, y]), jnp.array([CENTER, CENTER]))
        just_generated_orbiting = (tp_orbital + r) < ROTATION_RADIUS_LIMIT
        new_has_orbiting = has_orbiting | (success & just_generated_orbiting)

        return (new_planets, new_num, new_has_orbiting, key, attempts + 1)

    state_after_2 = lax.while_loop(phase_2_cond, phase_2_body, init_state_2)
    final_planets = state_after_2[0]

    return final_planets


In [50]:
generate_planets(jax.random.PRNGKey(42))

Array([[ 0.       , -1.       , 96.62998  , 82.71575  ,  1.       ,
        51.       ,  1.       ],
       [ 1.       , -1.       , 17.284248 , 96.62998  ,  1.       ,
        51.       ,  1.       ],
       [ 2.       , -1.       , 82.71575  ,  3.370018 ,  1.       ,
        51.       ,  1.       ],
       [ 3.       , -1.       ,  3.370018 , 17.284248 ,  1.       ,
        51.       ,  1.       ],
       [ 4.       , -1.       , 72.83497  , 97.2282   ,  2.3862944,
        43.       ,  4.       ],
       [ 5.       , -1.       ,  2.7717972, 72.83497  ,  2.3862944,
        43.       ,  4.       ],
       [ 6.       , -1.       , 97.2282   , 27.165031 ,  2.3862944,
        43.       ,  4.       ],
       [ 7.       , -1.       , 27.165031 ,  2.7717972,  2.3862944,
        43.       ,  4.       ],
       [ 8.       , -1.       , 96.51765  , 64.52269  ,  1.6931472,
        10.       ,  2.       ],
       [ 9.       , -1.       , 35.47731  , 96.51765  ,  1.6931472,
        10.       ,  2.

In [ ]:
# num = 5000
# base_dense = jnp.arange(num)
# t = 0.3 * jnp.pi + 1.4 * jnp.pi * base_dense / (num - 1)
# ex = c_val + a * jnp.cos(t)
# ey = b * jnp.sin(t)

In [53]:
a = jnp.arange(5)
b = jnp.arange(5) + 2
a, b

(Array([0, 1, 2, 3, 4], dtype=int32), Array([2, 3, 4, 5, 6], dtype=int32))

In [60]:
jnp.stack((a,b), axis=1)

Array([[0, 2],
       [1, 3],
       [2, 4],
       [3, 5],
       [4, 6]], dtype=int32)

In [1]:
!pip install jaxtyping

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.1 MB/s eta 0:00:00


In [1]:
import orbit_wars_jax
from jax.random import PRNGKey
a = orbit_wars_jax.setup(PRNGKey(0))

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [3]:
a = orbit_wars_jax.setup(PRNGKey(5))

In [6]:
!apt remove python3-blinker

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages were automatically installed and are no longer required:
  distro-info-data gir1.2-packagekitglib-1.0 libappstream4 libdw1
  libgstreamer1.0-0 libpackagekit-glib2-18 libpolkit-agent-1-0
  libpolkit-gobject-1-0 libstemmer0d libxmlb2 libyaml-0-2 lsb-release
  packagekit pkexec policykit-1 polkitd python-apt-common python3-apt
  python3-cffi-backend python3-cryptography python3-dbus python3-distro
  python3-gi python3-httplib2 python3-jeepney python3-jwt python3-keyring
  python3-lazr.uri python3-pkg-resources python3-pyparsing
  python3-secretstorage python3-six python3-wadllib
Use 'apt autoremove' to remove them.
The following packages will be REMOVED:
  python3-blinker python3-launchpadlib python3-lazr.restfulclient
  python3-oauthlib python3-software-properties software-properties-common
0 upgraded, 0 newly installed, 6 to remove and 1 not upgraded.
After this operat

In [7]:
!pip install --upgrade "kaggle-environments>=1.28.0"

  Using cached kaggle_environments-1.29.3-py3-none-any.whl.metadata (829 bytes)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached gymnax-0.0.8-py3-none-any.whl.metadata (19 kB)
  Using cached litellm-1.82.4-py3-none-any.whl.metadata (30 kB)
  Using cached open_spiel-1.6.15-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pettingzoo-1.24.0-py3-none-any.whl.metadata (8.1 kB)
  Using cached shimmy-2.0.1-py3-none-any.whl.metadata (4.4 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached openai-2.38.0-py3-none-any.whl.metadata (31 kB)
  Using cached tiktoken-0.13.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.7 kB)
Using cached kaggle_environments-1.29.3-py3-none-any.whl (13.4 MB)
Using cached gymnax-0.0.8-py3-none-any.whl (96 kB)
Using cached pettingzoo-1.24.0-py3-none-any.whl (840 kB)
Using cached flask-3.1.3-py3-none-any.whl (103 kB)
Using cached litellm-1.82.4-py3-none-any.whl (15.6

In [8]:
from kaggle_environments import make

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.


INFO:kaggle_environments.envs.open_spiel_env.open_spiel_env:Successfully loaded OpenSpiel environments: 24.


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons


INFO:kaggle_environments.envs.open_spiel_env.open_spiel_env:   open_spiel_amazons


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon


...

INFO:kaggle_environments.envs.open_spiel_env.open_spiel_env:   open_spiel_python_repeated_pokerkit


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: OpenSpiel games skipped: 1.


INFO:kaggle_environments.envs.open_spiel_env.open_spiel_env:OpenSpiel games skipped: 1.


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    snake


INFO:kaggle_environments.envs.open_spiel_env.open_spiel_env:   snake


In [10]:
env = make("orbit_wars", debug=False)